[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CityScope/UrbanAccessAnalyzer/blob/main/examples/walkability.ipynb)

# 🚶 Walkability

**How many everyday places can you reach on foot?** This notebook grades every street of a city by its walking distance to the places people need: shops, schools, health care, parks...

### How it works

1. **Streets and places** come from OpenStreetMap: the walking street network, and each type of place (POI) from the Overpass API.
2. **One grade per type of place.** From each POI we walk along the streets. A street scores **100** within the first distance step, and less and less up to the type's `max_distance`; beyond it, **0**.
3. **Points vs polygons.** Shops, schools... are *points*. Parks are *polygons*: people can enter anywhere along their edge, so every street touching the park (or inside it) counts as being at the park.
4. **One final score.** The grades are combined as a mean weighted by each type's `importance` (0 - 100).
5. **Who benefits?** The street scores are averaged into H3 hexagons and crossed with population.

| Config field | Meaning |
|---|---|
| `name` | label of the POI type |
| `query` | function that downloads the POIs from OSM |
| `geometry` | `"Point"` or `"Polygon"` |
| `importance` | relative weight in the final score |
| `max_distance` | meters walked beyond which the POI is worthless |

In [ ]:
# Google Colab only: installs everything (all packages pinned to release v1.0.0). Does nothing locally.
import sys

if "google.colab" in sys.modules:
    !pip install -q "UrbanAccessAnalyzer[plot] @ git+https://github.com/CityScope/UrbanAccessAnalyzer.git@v1.0.0"
    !pip install -q "geohierarchy @ git+https://github.com/CityScope/geohierarchy.git@v1.0.0"
    !pip install -q "pycensus @ git+https://github.com/CityScope/pyCensus.git@v1.0.0"
    # If an import fails afterwards: Runtime > Restart session, then run again.

In [ ]:
import os
import sys
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from UrbanAccessAnalyzer import api, grading, osm_io, plotting, utils

warnings.filterwarnings("ignore")

# 1 User inputs

## 1.1 Area of interest

In [ ]:
# USER INPUT
place = "Alcoy, España"

# Streets and POIs are downloaded for a larger area to avoid border effects (meters).
# Use at least the largest max_distance of the POIs below.
download_buffer = 1500

In [ ]:
# USER INPUT
results_path = os.path.normpath("output")

city_filename = utils.sanitize_filename(place)
city_results_path = os.path.join(results_path, city_filename)
os.makedirs(city_results_path, exist_ok=True)

pbf_path = os.path.join(city_results_path, "region.osm.pbf")  # downloaded from Geofabrik if missing
population_results_path = os.path.join(city_results_path, "population.gpkg")

maps_path = os.path.join(city_results_path, "maps")  # standalone .html maps: double-click to open
os.makedirs(maps_path, exist_ok=True)

In [ ]:
aoi = api.AreaOfInterest.from_name(place)  # Option 1: from the place name
# aoi = api.AreaOfInterest.from_file("my_area.gpkg")  # Option 2: your own file (.gpkg, .geojson, .shp)

utils.get_geographic_suggestions_from_string(place, user_agent="app")  # check that the right place was found

## 1.2 Points of interest

In [ ]:
# USER INPUT
# One entry per type of place. "icon" is the emoji shown on the maps (use any emoji you like).
# Ready-made queries in osm_io: schools, healthcare, groceries, shops, restaurants, libraries,
# pharmacies, gyms, cinemas, bus_stops, green_areas. Your own data or OSM tags: see 1.3.
pois = [
    {"name": "schools",     "query": osm_io.schools,     "geometry": "Point",   "importance": 10, "max_distance": 1000, "icon": "🏫"},
    {"name": "groceries",   "query": osm_io.groceries,   "geometry": "Point",   "importance": 8,  "max_distance": 1500, "icon": "🛒"},
    {"name": "restaurants", "query": osm_io.restaurants, "geometry": "Point",   "importance": 7,  "max_distance": 1500, "icon": "🍽️"},
    {"name": "healthcare",  "query": osm_io.healthcare,  "geometry": "Point",   "importance": 3,  "max_distance": 2000, "icon": "🏥"},
    {"name": "parks",       "query": osm_io.green_areas, "geometry": "Polygon", "importance": 5,  "max_distance": 1500, "icon": "🌳"},
    # Any other OSM tag: osm_io.tag_query(key, value)
    {"name": "bakeries",    "query": osm_io.tag_query("shop", "bakery"), "geometry": "Point", "importance": 4, "max_distance": 1000, "icon": "🥖"},
]

## 1.3 Your own points of interest (optional)

Another type of place is just one more entry in `pois`. Two ways to get its data:

### A. Your own file

Any file with **points or polygons** that GeoPandas reads: GeoPackage (`.gpkg`), GeoJSON, Shapefile (`.shp` + its sidecar files, zipped as `.zip`), GeoParquet... in **any coordinate system** (it must have one). An optional column holds the names shown on the maps.

- **Running locally:** put the file next to the notebook (or use its full path) and set `my_file`.
- **Google Colab:** the notebook runs on a remote machine, so your computer's files are not visible. Upload the file first, either
  1. set `upload_in_colab = True` below and run the cell: a file picker opens and the file is saved in `/content/`; or
  2. click the 📁 **Files** icon in the left sidebar, then the upload icon (⬆️), choose the file, and set `my_file = "my_pois.gpkg"`; or
  3. keep it in Google Drive: `from google.colab import drive; drive.mount("/content/drive")` and use the path `/content/drive/MyDrive/my_pois.gpkg`.

  Colab deletes uploaded files when the session ends.

In [ ]:
# USER INPUT
my_file = ""  # path to your file, e.g. "my_pois.gpkg". "" = don't use one
upload_in_colab = False  # Colab only: True opens a file picker to upload it from your computer

if upload_in_colab and "google.colab" in sys.modules:
    from google.colab import files

    my_file = next(iter(files.upload()))  # uploaded files are saved in /content/

if my_file:
    pois.append({
        "name": "my_pois",
        "query": grading.query_from_file(my_file, name_column=None),  # name_column: column with the names, if any
        "geometry": "Point",  # "Polygon" if your file has polygons (parks, plazas...)
        "importance": 5,
        "max_distance": 1000,
        "icon": "⭐",
    })

### B. A custom OpenStreetMap query

OpenStreetMap describes every place with tags (`key=value`). To find the tags of what you want, browse the [map features](https://wiki.openstreetmap.org/wiki/Map_features) or search a value on [taginfo](https://taginfo.openstreetmap.org/), or click any place on [openstreetmap.org](https://www.openstreetmap.org) with the *query features* tool.

- **One tag:** `osm_io.tag_query("amenity", "dentist")`. Use `"*"` as the value for any value.
- **Anything else:** write the [Overpass](https://overpass-turbo.eu/) query yourself. `{{bbox}}` is replaced by the download area. Tip: ask an AI assistant *"write an Overpass query for hospitals and clinics, using {{bbox}}"*.

In [ ]:
def pharmacies_or_dentists(bounds):
    query = """
        [out:json][timeout:25];
        nwr["amenity"~"pharmacy|dentist"]({{bbox}});
        (._;>;);
        out body;
    """
    return osm_io.overpass_query(query, bounds)


# Uncomment to add it (nwr = node, way and relation; use "Polygon" for big areas such as parks)
# pois.append({"name": "pharmacies_dentists", "query": pharmacies_or_dentists, "geometry": "Point",
#              "importance": 3, "max_distance": 1000, "icon": "💊"})

## 1.4 Parameters

In [ ]:
# USER INPUT
n_steps = 5  # distance steps per POI type: with 5, scores are 100, 80, 60, 40, 20, then 0
min_edge_length = 30  # meters, simplifies the street graph
h3_resolution = 9  # output hexagon size (9 is ~0.1 km2, 10 is ~0.015 km2)
show_maps = True  # maps take some time to render

In [ ]:
# USER INPUT
# "worldpop": WorldPop 100 m raster, works anywhere in the world.
# "census_spain": official INE census sections (secciones censales), Spain only.
population_source = "worldpop"

# 2 Processing

## 2.1 Street network

In [ ]:
aoi.gdf = gpd.GeoDataFrame(geometry=[aoi.gdf.union_all()], crs=aoi.gdf.crs)  # a single polygon

# Bigger area for downloading streets and POIs, to avoid border effects
utm = aoi.gdf.estimate_utm_crs()
aoi_download = api.AreaOfInterest(
    gpd.GeoDataFrame(geometry=aoi.gdf.to_crs(utm).buffer(download_buffer), crs=utm).to_crs(aoi.gdf.crs)
)

network = api.StreetNetwork.from_pbf(pbf_path, aoi=aoi_download, network_type="walk", simplify_distance=min_edge_length)
print(f"{network.nodes.height} nodes, {network.edges.height} edges")

## 2.2 Points of interest

In [ ]:
poi_gdfs = grading.fetch_pois(pois, aoi_download.gdf, crs=network.crs)  # if it fails, run again: Overpass can be busy

In [ ]:
m = None
if show_maps:
    m = plotting.general_map(
        aoi=aoi.gdf, pois=grading.poi_table(pois, poi_gdfs), poi_column="type", poi_icon_column="icon"
    )
    m.save(os.path.join(maps_path, "points_of_interest.html"))  # double-click it to open it in the browser
m

## 2.3 Grade the streets

In [ ]:
edges_gdf, network = grading.grade(network, pois, poi_gdfs, n_steps=n_steps, min_edge_length=min_edge_length)
edges_gdf = edges_gdf[edges_gdf.intersects(aoi.gdf.to_crs(network.crs).union_all())]  # keep the study area only
edges_gdf.to_file(os.path.join(city_results_path, "access_streets.gpkg"))
edges_gdf[[p["name"] for p in pois] + ["access_score"]].describe().round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
edges_gdf.plot(ax=ax, column="access_score", cmap="RdYlGn", vmin=0, vmax=100, linewidth=1.5, legend=True,
               legend_kwds={"label": "walkability (0-100)", "shrink": 0.6})
ax.set_title(f"Walkability: {place}")
ax.set_axis_off()
plt.show()

# 3 Population

In [ ]:
from geohierarchy import GeoHierarchy, Sum, Mean
from geohierarchy.utils import h3_cells

hierarchy = GeoHierarchy()
hierarchy.add_level("h3", h3_cells(aoi.gdf, resolution=h3_resolution), id_col="h3")
# Streets are lines: average their score inside each hexagon, weighted by length
hierarchy.add_vector_data(
    edges_gdf, level="h3", columns=[*[p["name"] for p in pois], "access_score"], agg=Mean(), geoweight_by="length",
)

# Population is a count: add it up inside each hexagon
if population_source == "worldpop":
    import pycensus.worldwide.worldpop as worldpop

    population_file = worldpop.download_worldpop_population(
        aoi.gdf, 2025, folder=results_path, resolution="100m"
    )

    # WorldPop files cover a whole country: keep only the study area (saves a lot of memory)
    # and turn the "no data" value into NaN so it is not summed.
    import rasterio
    from rasterio.windows import from_bounds

    with rasterio.open(population_file) as src:
        win = from_bounds(*aoi.gdf.to_crs(src.crs).total_bounds, transform=src.transform).round_offsets().round_lengths()
        values = src.read(1, window=win).astype("float32")
        values[values == src.nodata] = np.nan
        profile = src.profile | {"height": win.height, "width": win.width, "transform": src.window_transform(win),
                                  "tiled": False, "nodata": np.nan, "count": 1}
        profile.pop("blockxsize", None)
        profile.pop("blockysize", None)
        population_file = os.path.join(city_results_path, "population.tif")
        with rasterio.open(population_file, "w", **profile) as dst:
            dst.write(values, 1)
    hierarchy.add_raster_data(population_file, column="population", level="h3", agg=Sum())
elif population_source == "census_spain":
    from pycensus.countries.spain.ine import loader

    sections = loader.load(aoi=aoi.gdf, level="section")  # one polygon per census section
    hierarchy.add_vector_data(
        sections, level="h3", columns=["population"], agg=Sum(), geoweight_by="area",
    )
else:
    raise ValueError(f"Unknown population_source: {population_source!r}")

results_h3_gdf = hierarchy["h3"]
results_h3_gdf = results_h3_gdf[results_h3_gdf.intersects(aoi.gdf.union_all())]
results_h3_gdf.to_file(population_results_path)
results_h3_gdf.head()

In [ ]:
labels = {"population": "👥 Population", "access_score": "🚶 Walkability (0-100)", **{p["name"]: f"{p.get('icon', '📍')} {p['name']} (0-100)" for p in pois}}  # data column -> name shown when you hover over the map
max_street_features = 10000  # streets are drawn only if there are fewer (keeps the .html small)

results_h3_gdf["population"] = results_h3_gdf["population"].fillna(0).round().astype(int)
hex_view = results_h3_gdf[[*labels, "geometry"]].rename(columns=labels).round(1)
layers = [hex_view]
if len(edges_gdf) <= max_street_features:
    street_cols = [c for c in labels if c in edges_gdf.columns]
    layers.append(edges_gdf.to_crs(4326)[[*street_cols, "geometry"]].rename(columns=labels).round(1))
else:
    print(f"{len(edges_gdf)} streets: too many for the map, showing hexagons only (streets are in access_streets.gpkg)")

m = None
if show_maps:
    m = plotting.general_map(
        aoi=aoi.gdf, gdfs=layers, column=labels["access_score"], cmap="RdYlGn", vmin=0, vmax=100,
        pois=grading.poi_table(pois, poi_gdfs), poi_column="type", poi_icon_column="icon",
    )
    m.save(os.path.join(maps_path, "walkability_map.html"))  # double-click it to open it in the browser
m

# 4 Statistics

In [ ]:
bins = [0, 20, 40, 60, 80, 100.01]
labels = ["0-20", "20-40", "40-60", "60-80", "80-100"]
results_h3_gdf["walkability"] = pd.cut(results_h3_gdf["access_score"], bins=bins, labels=labels, right=False)

stats_df = results_h3_gdf.groupby("walkability", observed=False, as_index=False)["population"].sum()
total_population = stats_df["population"].sum()
stats_df["population %"] = (stats_df["population"] * 100 / total_population).round(1)
stats_df["population"] = stats_df["population"].round(0).astype(int)
stats_df.to_csv(os.path.join(city_results_path, "stats.csv"), index=False)
stats_df

In [ ]:
if "google.colab" in sys.modules:
    !zip -r /content/output.zip "{results_path}"  # then download it from the file browser

Output files in `output/<place>/`:

- `maps/points_of_interest.html` and `maps/walkability_map.html`: interactive maps, double-click to open them in your browser
- `access_streets.gpkg`: streets with one score per POI type and the final `access_score`
- `population.gpkg`: H3 hexagons with population and mean score
- `stats.csv`: population per score band